# transformer-residual-lab · Kaggle 低位宽量化验证台（v1.2.0）

**只需要改一个开关**（下面第一个代码格），然后 `Run All`。

| 开关 | 会做什么 | 预估 |
|---|---|---|
| `RUN_EVAL = True` | **只评测**：从 HF 已存的 GGUF 取输入，跑 S6 配对 NLL + S7 质量检查。不下载 BF16、不重做 S3、不重新量化；**缺 `llama-perplexity`/`llama-cli` 时自动只编 S2**（不下载模型）；**有 GPU 就用 GPU**，一次只装一个模型 | GPU 上分钟级（假设速率，非实测）；需补编时 +≈28 分钟 |
| `RUN_S2 = True` | 编译 llama.cpp（下载 36.6 MiB 源码 → cmake → 4 个二进制） | ≈28 分钟 |
| `RUN_S3 = True` | 自动带上 S2，再生成 `imatrix.gguf`（下 8.11 GiB Q8_0 + wikitext，算完回收 Q8_0） | +≈10 分钟 |
| `RUN_S4 = True` | 自动带上 S2+S3，再从**原始 BF16** 量化 IQ2_XS / IQ4_XS，并统计真实字节 | +≈30–60 分钟 |

全部 `False` 时只跑 S0/S1/S9（环境检查 + 冒烟 + 报告），**不联网、不下载任何东西**。
打开任一开关前请确认右侧 `Settings → Internet = On`。

`RUN_EVAL = True` 需要 HF 私有仓库的 token（`HF_TOKEN` 环境变量 / Kaggle secret /
本机 `huggingface-cli login` 的缓存，三者任一）。**不要把 token 写进 notebook。**
Kaggle 上要在 `Add-ons → Secrets` 新建 `HF_TOKEN` 并**勾选 attach 到本 notebook**；
配一次就够。没有 token 时它会在**编译之前**停下（3 秒内），`out/FAILURE.json` 会同时
列出"没 token"和"缺二进制"两条根因，不用去翻日志。
它会把每个阶段的小结果（5 个 JSON/MD，不含模型、不含日志）出站到 HF 并远端复核；
想彻底不外发就 `--set HF_UPLOAD_AFTER_RUN=false`。

**新 Session 不需要你额外开 S2**：`RUN_EVAL = True` 会先探活
`llama-perplexity` / `llama-cli`，缺了就**自己只编 S2**（pinned 源码 36.6 MiB，
不下载 Q8_0/BF16、不做 S3/S4、不重量化）；都在就直接进评测。

**v1.2.0：S6 不再"见 1 行文本重叠就整轮归零"**。校准集（wikitext **train** 全文）
与评测集（**test**）里出现相同句子，不等于 imatrix 实际看过评测数据 ——
imatrix 只消费 `32×512 ≈ 16.4k` token。现在 S6 会：给每条交集做取证
（完整哈希 / 段索引 / 是否只是前 200 字符相同）→ 剔除共享前缀的评测段，
生成确定性的 `eval_holdout.txt`（两个量化共用同一份，规则与剔除清单写在
`out/isolation/isolation.json`）→ 在结论里如实标注
`actual_imatrix_calib_overlap`（证明不了就写 `unknown`，不假称无污染）。
**你不需要改任何配置**；只有当剔除比例 >10%（像把 train 当成了 eval）时它才会停下。

**Kaggle 是可丢弃的算力机**：跑失败就直接关掉 Session，修代码 → commit → 新 Session
重跑，输入不用重下。失败现场写在 `out/FAILURE.json`（最早根因不会被后面的失败覆盖），
不需要 Save Version 或抢救工作区。**S6/S7 没出数据时报告顶部会写"本次运行失败"**，
不会用一份看起来正常的报告掩盖过去。

`RUN_S4 = True` 会在**任何编译、任何下载之前**先跑一次几秒的 `S4_PREFLIGHT`：
只读核查 working / 工作区外暂存盘的挂载、容量、写权限，以及 BF16 有没有落点、
imatrix 能不能复用。不通过就**就地停下**并只给一步要做的 —— 不会先花 28 分钟编译、
再下 8 GiB / 15 GiB 之后才发现磁盘不够。

**代码不在这份 notebook 里。** 第二个代码格会从公开仓库按**钉死的 commit**
sparse fetch `research/kaggle/trlab/`（约 200 KB，1–3 秒），校验 `git rev-parse HEAD`
与钉的值一致后才 import。源码在
[https://github.com/Atomheart-Father/transformer-residual-lab](https://github.com/Atomheart-Father/transformer-residual-lab/tree/main/research/kaggle/trlab)，
单元测试直接 import 它，不解析 notebook。

S8 / SX 本版不启动；产物与 provenance 落在 `/kaggle/working/trlab/out/`。


## 开关

默认全部 `False`。想跑哪一级就改哪一个：`RUN_EVAL` 只评测（不动模型，
缺二进制会自动只编 S2），`RUN_S4` 会自动把 S2/S3 需要的东西都带上。

In [ ]:
# 只需要改这一格。全部 False = 不联网、不下载（只跑 S0/S1/S9）。
RUN_EVAL = False    # ← 只评测（从 HF 已存 GGUF 取输入；缺二进制会自动只编 S2；有 GPU 就用 GPU）
RUN_S2 = False
RUN_S3 = False
RUN_S4 = False      # ← 想量化 IQ2_XS / IQ4_XS 就改成 True


## 执行

这一格负责：取执行包 → 校验 → `trlab.run(...)`。失败会停在这一格并说清原因，
不会带着残缺状态往下跑。

只想做那个几秒的磁盘检查、不编译不下载：把最后一行改成
`trlab.run(preflight_only=True)`。`RUN_S4 = True` 的 `Run All` 已经**自动**先跑它，
不用自己先单独点一轮。

预检不通过时这一格会直接停下，并只给一步要做的（例如先做 `trlab_fetch_bf16.ipynb`
把 BF16 变成只读 Input，或清掉 working 里的旧产物）；明细落在
`/kaggle/working/trlab/out/S4_PREFLIGHT.json`。

**只评测（`RUN_EVAL = True`）**：输入来自 HF 私有仓库里已存的三个 GGUF，
不下载 15 GiB BF16、不重做 S3、不重新量化；**本 Session 缺 `llama-perplexity` /
`llama-cli` 时它会自动只编 S2**（pinned 源码 36.6 MiB，不下载模型），所以你不需要
再开第二个开关。缺 token 时会在**编译之前**停下，`out/FAILURE.json` 同时给
"没 token"与"缺二进制"两条根因，`next_step` 只写一件事：去 `Add-ons → Secrets`
加 `HF_TOKEN` 并 attach，再 `Run all` —— **不要把 token 写进 notebook**。
如果看到的是 **HTTP 400**，那是本工具请求参数的问题（tree API），
**不是你的 token**，换最新 notebook 重跑即可，不要重建 token。
**有 GPU 就用 GPU**（`-ngl 99`，装不下才退 CPU 并把原因写进 `device_by_fmt`）；
一次只装一个模型，跑完释放再装下一个。
每个阶段的小结果会自动出站到 HF（5 个 JSON/MD）并做远端复核；失败时这一格
不会假装成功，现场在 `out/FAILURE.json`，报告顶部会写"本次运行失败"。

S8 / SX 默认关闭。要跑就把最后一行改成
`trlab.run(run_eval=True, S8_BF16_ANCHOR=True)` —— 阶段名直接当关键字参数传。

In [ ]:
# =====================================================================
# 自动获取执行包并校验，然后跑。你不需要改这一格的任何东西。
# =====================================================================
import subprocess as _sp, sys as _sys, shutil as _shutil
from pathlib import Path as _P

SRC_URL     = "https://github.com/Atomheart-Father/transformer-residual-lab"
SRC_COMMIT  = "358bd3b4c908f8d7d26d514317a36a73870c375f"      # 钉死：GitHub 上 commit 不可变，可强校验
PKG_RELPATH = "research/kaggle/trlab"
_KAGGLE     = _P("/kaggle/working").is_dir()
DEST        = _P("/kaggle/working/_trlab_src") if _KAGGLE else _P.cwd() / "_trlab_src"
STAMP       = DEST / ".trlab_commit"

def _git(*args, **kw):
    return _sp.run(["git", *args], cwd=str(DEST), capture_output=True,
                   text=True, timeout=120, **kw)

def _already_here():
    """同一 commit 的包已经在盘上就直接复用（同一 Session 里重复 Run All 不重下）。"""
    if not (DEST / PKG_RELPATH / "__init__.py").exists():
        return False
    try:
        return STAMP.read_text().strip() == SRC_COMMIT
    except OSError:
        return False

def _fetch_from_github():
    """钉 commit + sparse checkout：只取 research/kaggle/trlab 这一个子目录。"""
    if not _shutil.which("git"):
        raise RuntimeError("环境里没有 git，无法获取执行包")
    DEST.mkdir(parents=True, exist_ok=True)
    for attempt in (1, 2, 3):
        r = _git("init", "-q")
        r = _git("config", "core.sparseCheckout", "true")
        r = _git("sparse-checkout", "set", PKG_RELPATH)
        r = _git("remote", "add", "origin", SRC_URL)
        r = _git("fetch", "--depth", "1", "--filter=blob:none",
                 "origin", SRC_COMMIT)
        if r.returncode == 0:
            break
        if attempt == 3:
            raise RuntimeError(
                "取执行包失败（试了 3 次）。最后一条 git 输出：\n" + (r.stderr or "")[-600:]
                + "\n常见原因：仓库不是 public、commit 不存在、或网络不通。")
        _git("remote", "remove", "origin")
    r = _git("checkout", "-q", "FETCH_HEAD")
    if r.returncode != 0:
        raise RuntimeError("checkout 失败：" + (r.stderr or "")[-400:])
    got = _git("rev-parse", "HEAD").stdout.strip()
    if got != SRC_COMMIT:
        raise RuntimeError(f"校验失败：拿到 {got}，期望 {SRC_COMMIT}")
    STAMP.write_text(SRC_COMMIT + "\n")
    return DEST / PKG_RELPATH

def _bootstrap():
    if _already_here():
        print(f"执行包已在盘上（commit {SRC_COMMIT[:12]}），直接复用")
        return DEST / PKG_RELPATH
    try:
        pkg = _fetch_from_github()
    except Exception as e:
        # 仓库取不到（不是 public / commit 被改 / 网络不通）时，
        # 退回 notebook 内嵌的同一份执行包（sha256 校验后解压）。
        if "_fetch_from_embedded" not in globals():
            raise
        print(f"从仓库取包失败：{e}")
        print("改用 notebook 内嵌的执行包（sha256 校验后解压）")
        pkg = _fetch_from_embedded()
    print(f"执行包就绪：{SRC_COMMIT[:12]}  {pkg}")
    return pkg

_pkg_parent = str(_bootstrap().parent)
if _pkg_parent not in _sys.path:
    _sys.path.insert(0, _pkg_parent)

import trlab
print(f"trlab v{trlab.__version__}  运行位置：{'Kaggle' if _KAGGLE else '本地/其他'}")

trlab.run(run_s2=RUN_S2, run_s3=RUN_S3, run_s4=RUN_S4, run_eval=RUN_EVAL)
